# 1. 단일 Lakehouse 비교 테스트: 원본 복사

기존 UI4 모델과 같은 4개 테이블을 **새 테스트 Lakehouse**에 복사합니다. 원본 MES/QMS, null, 컬럼 타입, 행은 변경하지 않습니다. 새 데이터를 생성하거나 필터링하지 않습니다.

## 준비와 실행

- Fabric Spark 노트북으로 업로드합니다. 작업 영역 Contributor 이상, 원본 읽기, Ontology Preview 사용 권한이 필요합니다. NotebookUtils 실행자 인증을 사용하며 비밀값 입력은 없습니다.
- 설정의 원본 ID는 이번 실습에서 확인한 값입니다. 본인 대상이 아니면 변경합니다. `SOURCE_ONTOLOGY_ID`는 **Equipment, ProcessRun, Inspection, Nonconformance와 관계 5개를 가진 UI4** 항목입니다. 해당 UI4가 삭제됐다면 진행할 수 없습니다.
- **Run all 대신 셀별 실행:** 1~4번 셀까지 실행해 테스트 Lakehouse와 기준선 파일을 만듭니다. 출력된 새 Lakehouse를 기본으로 Attach한 뒤 6번 복사 셀을 실행합니다.
- Attach 과정에서 세션이 재시작되면 2번 설정 셀의 `TEST_LAKEHOUSE_ID`에 생성된 ID를 입력하고 **2번 셀만** 재실행합니다. 3~4번은 건너뛰고 6번으로 진행합니다.
- 동명 Lakehouse, 기존 대상 테이블/매니페스트가 있으면 중단합니다. 부분 실패 후 무조건 재실행하지 말고 테스트 항목을 확인하거나 새 이름을 사용합니다.
- Spark, Lakehouse 저장소와 후속 Graph 적재에 용량 비용이 발생합니다. 로컬 문법 검증은 Fabric 실행 검증을 대신하지 않습니다.

완료 후 [두 번째 노트북](02_create_single_lakehouse_ontology.ipynb)에 출력된 테스트 Lakehouse ID를 입력합니다. 두 노트북은 세션을 공유하지 않으며 `Files/test_setup/manifest.json`으로 설정·원본 정의·검증 결과를 전달합니다.

정리는 두 번째 노트북 안내를 따릅니다. 원본 Lakehouse는 삭제하지 않습니다.

In [ ]:
import base64
import json
import re
import time
import uuid
from urllib.parse import urlsplit

import notebookutils
import requests

WORKSPACE_ID = str(uuid.UUID(notebookutils.runtime.context['currentWorkspaceId']))
MES_LAKEHOUSE_ID = 'e310e555-d29a-429d-8001-4d046f2e120b'
QMS_LAKEHOUSE_ID = '08a23c63-6a53-41b6-9238-beffc30da8d3'
SOURCE_ONTOLOGY_ID = '4424f2ae-c6d6-4060-ba75-1fb320ae0e06'
TEST_LAKEHOUSE_NAME = 'mes_qms_single_test'
TEST_LAKEHOUSE_ID = ''
API_ROOT = 'https://api.fabric.microsoft.com/v1'


def api_call(method, path, body=None):
    url = API_ROOT + path if path.startswith('/') else path
    parsed = urlsplit(url)
    if parsed.scheme != 'https' or parsed.netloc != 'api.fabric.microsoft.com' or not parsed.path.startswith('/v1/'):
        raise ValueError('Unexpected Fabric API URL')
    for attempt in range(5):
        try:
            response = requests.request(method, url, json=body, headers={
                'Authorization': 'Bearer ' + notebookutils.credentials.getToken('pbi'),
            }, timeout=90, allow_redirects=False)
        except requests.RequestException:
            raise RuntimeError('Fabric transport failure; inspect existing items before retrying a create.') from None
        with response:
            headers = dict(response.headers)
            payload = response.json() if response.content else None
            status_code = response.status_code
        if status_code == 429 and attempt < 4:
            time.sleep(min(float(headers.get('Retry-After', 10)), 60))
            continue
        if status_code not in (200, 201, 202, 204):
            raise RuntimeError(f'Fabric HTTP {status_code}: {json.dumps(payload, ensure_ascii=False)}')
        return status_code, payload, headers
    raise RuntimeError('Fabric throttling retry limit exceeded')


def operation(method, path, body=None):
    status_code, payload, headers = api_call(method, path, body)
    if status_code != 202:
        return payload
    operation_id = str(uuid.UUID(headers['x-ms-operation-id']))
    deadline = time.monotonic() + 1800
    while time.monotonic() < deadline:
        time.sleep(min(float(headers.get('Retry-After', 5)), 60))
        _, state, headers = api_call('GET', f'/operations/{operation_id}')
        if state['status'] == 'Succeeded':
            return api_call('GET', f'/operations/{operation_id}/result')[1]
        if state['status'] not in ('NotStarted', 'Running'):
            raise RuntimeError(f'Fabric operation failed: {state}')
    raise TimeoutError('Operation completion unknown; inspect the item before retrying.')


def list_items(path):
    rows, visited = [], set()
    while path:
        if path in visited:
            raise RuntimeError('Repeated pagination URL')
        visited.add(path)
        payload = api_call('GET', path)[1]
        rows.extend(payload['value'])
        path = payload.get('continuationUri')
        if payload.get('continuationToken') and not path:
            raise RuntimeError('Pagination URI missing; refusing an incomplete list')
    return rows


def decode_definition(definition):
    parts = {}
    for part in definition['parts']:
        if part['payloadType'] != 'InlineBase64' or part['path'] in parts:
            raise ValueError('Unsupported or duplicate definition part')
        parts[part['path']] = json.loads(base64.b64decode(part['payload'], validate=True))
    return parts


def table_path(item_id, table):
    return f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{item_id}/Tables/dbo/{table}'


EXPECTED_ENTITIES = {
    'Equipment': ('mes_equipment', 'eqp_id', MES_LAKEHOUSE_ID),
    'ProcessRun': ('mes_process_result', 'id', MES_LAKEHOUSE_ID),
    'Inspection': ('qms_inspection', 'inspection_id', QMS_LAKEHOUSE_ID),
    'Nonconformance': ('qms_nonconformance', 'ncr_id', QMS_LAKEHOUSE_ID),
}
EXPECTED_RELATIONS = {
    'EquipmentHasRuns': ('Equipment', 'ProcessRun', 'mes_process_result', 'eqp_id', 'id'),
    'RunHasInspection': ('ProcessRun', 'Inspection', 'qms_inspection', 'mes_process_result_id', 'inspection_id'),
    'EquipmentHasInspection': ('Equipment', 'Inspection', 'qms_inspection', 'eqp_id', 'inspection_id'),
    'EquipmentHasNcr': ('Equipment', 'Nonconformance', 'qms_nonconformance', 'eqp_id', 'ncr_id'),
    'InspectionHasNcr': ('Inspection', 'Nonconformance', 'qms_nonconformance', 'inspection_id', 'ncr_id'),
}
for identifier in (MES_LAKEHOUSE_ID, QMS_LAKEHOUSE_ID, SOURCE_ONTOLOGY_ID):
    uuid.UUID(identifier)
if MES_LAKEHOUSE_ID == QMS_LAKEHOUSE_ID:
    raise ValueError('The baseline must use separate MES and QMS Lakehouses')
if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,79}', TEST_LAKEHOUSE_NAME):
    raise ValueError('Use a simple test Lakehouse name')
print('Workspace:', WORKSPACE_ID)
print('Source MES/QMS:', MES_LAKEHOUSE_ID, QMS_LAKEHOUSE_ID)
print('UI4 ontology:', SOURCE_ONTOLOGY_ID)
print('New Lakehouse:', TEST_LAKEHOUSE_NAME)

In [ ]:
BASELINE = decode_definition(operation('POST', f'/workspaces/{WORKSPACE_ID}/ontologies/{SOURCE_ONTOLOGY_ID}/getDefinition')['definition'])
ENTITIES = {body['name']: body for path, body in BASELINE.items()
            if path.startswith('EntityTypes/') and path.endswith('/definition.json')}
RELATIONS = {body['name']: body for path, body in BASELINE.items()
             if path.startswith('RelationshipTypes/') and path.endswith('/definition.json')}
if set(ENTITIES) != set(EXPECTED_ENTITIES) or set(RELATIONS) != set(EXPECTED_RELATIONS):
    raise ValueError('Source ontology must be the UI4 baseline: exactly 4 entities and 5 relationships')

for entity_name, (table, key, item_id) in EXPECTED_ENTITIES.items():
    entity = ENTITIES[entity_name]
    bindings = [body['dataBindingConfiguration'] for path, body in BASELINE.items()
                if path.startswith(f"EntityTypes/{entity['id']}/DataBindings/")]
    if len(bindings) != 1 or bindings[0]['dataBindingType'] != 'NonTimeSeries':
        raise ValueError(f'{entity_name}: expected one static binding and no telemetry')
    binding = bindings[0]
    expected_source = dict(workspaceId=WORKSPACE_ID, itemId=item_id, sourceSchema='dbo', sourceTableName=table, sourceType='LakehouseTable')
    if any(binding['sourceTableProperties'].get(field) != value for field, value in expected_source.items()):
        raise ValueError(f'{entity_name}: unexpected source location')
    key_bindings = [mapping['targetPropertyId'] for mapping in binding['propertyBindings'] if mapping['sourceColumnName'] == key]
    if key_bindings != entity['entityIdParts']:
        raise ValueError(f'{entity_name}: unexpected key mapping')

for name, (origin, target, table, origin_column, target_column) in EXPECTED_RELATIONS.items():
    relation = RELATIONS[name]
    bindings = [body for path, body in BASELINE.items()
                if path.startswith(f"RelationshipTypes/{relation['id']}/Contextualizations/")]
    if len(bindings) != 1:
        raise ValueError(f'{name}: expected one contextualization')
    binding = bindings[0]
    source_item = MES_LAKEHOUSE_ID if table.startswith('mes_') else QMS_LAKEHOUSE_ID
    expected_source = dict(workspaceId=WORKSPACE_ID, itemId=source_item, sourceSchema='dbo', sourceTableName=table, sourceType='LakehouseTable')
    if any(binding['dataBindingTable'].get(field) != value for field, value in expected_source.items()):
        raise ValueError(f'{name}: unexpected mapping table')
    for side, entity_name, column in [('source', origin, origin_column), ('target', target, target_column)]:
        entity = ENTITIES[entity_name]
        expected = [{'sourceColumnName': column, 'targetPropertyId': entity['entityIdParts'][0]}]
        if relation[side]['entityTypeId'] != entity['id'] or binding[side + 'KeyRefBindings'] != expected:
            raise ValueError(f'{name}: unexpected endpoint/key binding')

for item_id in (MES_LAKEHOUSE_ID, QMS_LAKEHOUSE_ID):
    lakehouse = api_call('GET', f'/workspaces/{WORKSPACE_ID}/lakehouses/{item_id}')[1]
    if lakehouse['properties'].get('defaultSchema') != 'dbo':
        raise ValueError('Source Lakehouses must be schema-enabled with dbo')

if any(item['displayName'].casefold() == TEST_LAKEHOUSE_NAME.casefold()
       for item in list_items(f'/workspaces/{WORKSPACE_ID}/lakehouses')):
    raise RuntimeError('Test Lakehouse name already exists. Do not overwrite; inspect it or choose a new name.')
created = operation('POST', f'/workspaces/{WORKSPACE_ID}/lakehouses', {
    'displayName': TEST_LAKEHOUSE_NAME,
    'description': 'UI4 single Lakehouse comparison; original data is not modified.',
    'creationPayload': {'enableSchemas': True},
})
TEST_LAKEHOUSE_ID = str(uuid.UUID(created['id']))
print('Created TEST_LAKEHOUSE_ID:', TEST_LAKEHOUSE_ID)
print('STOP: Attach this new Lakehouse as the default before running the copy cell.')
print('If the session restarts: enter this ID in the settings cell, rerun that cell, skip this creation cell.')

In [ ]:
SETUP_ROOT = f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{TEST_LAKEHOUSE_ID}/Files/test_setup'
notebookutils.fs.mkdirs(SETUP_ROOT)
notebookutils.fs.put(SETUP_ROOT + '/baseline.json', json.dumps({
    'workspace_id': WORKSPACE_ID, 'test_lakehouse_id': TEST_LAKEHOUSE_ID,
    'source_ontology_id': SOURCE_ONTOLOGY_ID, 'mes_id': MES_LAKEHOUSE_ID, 'qms_id': QMS_LAKEHOUSE_ID,
    'definition': BASELINE,
}, ensure_ascii=False), False)
print('Baseline saved. Now attach the new test Lakehouse as default.')

## 여기서 잠시 중단: 새 테스트 Lakehouse를 기본으로 Attach

**위 4개 셀까지 실행한 후** 출력된 테스트 Lakehouse를 이 노트북의 기본 Lakehouse로 연결합니다. 원본 MES/QMS를 기본으로 연결한 상태에서는 아래 복사 셀이 중단합니다.

세션이 재시작되면 2번 설정 셀의 `TEST_LAKEHOUSE_ID`에 생성된 ID를 넣고 **2번 셀만** 다시 실행합니다. 3~4번 생성·기준선 저장 셀은 재실행하지 않습니다. 아래 셀은 저장된 기준선을 다시 읽습니다.

복사는 `errorifexists`로 수행합니다. 중간에 실패해 일부 테이블만 생겼다면 자동 삭제하지 않습니다. 테스트 Lakehouse의 내용과 출력된 ID를 확인한 뒤 별도 이름으로 다시 시작하세요.

In [ ]:
from pyspark.sql import functions as F

TEST_LAKEHOUSE_ID = str(uuid.UUID(TEST_LAKEHOUSE_ID))
context = notebookutils.runtime.context
if TEST_LAKEHOUSE_ID in (MES_LAKEHOUSE_ID, QMS_LAKEHOUSE_ID):
    raise ValueError('Refusing to write to an original Lakehouse')
if (context.get('defaultLakehouseId') != TEST_LAKEHOUSE_ID
        or context.get('defaultLakehouseWorkspaceId') != WORKSPACE_ID):
    raise ValueError('Attach the new test Lakehouse as default, then restart the session if needed')
metadata = api_call('GET', f'/workspaces/{WORKSPACE_ID}/lakehouses/{TEST_LAKEHOUSE_ID}')[1]
if metadata['displayName'] != TEST_LAKEHOUSE_NAME or metadata['properties'].get('defaultSchema') != 'dbo':
    raise ValueError('Test Lakehouse name/schema mismatch')
SETUP_ROOT = f'abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{TEST_LAKEHOUSE_ID}/Files/test_setup'
recipe = json.loads(notebookutils.fs.head(SETUP_ROOT + '/baseline.json', 1048576))
if (recipe['workspace_id'], recipe['test_lakehouse_id'], recipe['mes_id'], recipe['qms_id']) != (WORKSPACE_ID, TEST_LAKEHOUSE_ID, MES_LAKEHOUSE_ID, QMS_LAKEHOUSE_ID):
    raise ValueError('Saved baseline does not match settings')
if notebookutils.fs.exists(SETUP_ROOT + '/manifest.json'):
    raise RuntimeError('Copy already completed; use notebook 02')
for table, _, _ in EXPECTED_ENTITIES.values():
    if spark.catalog.tableExists(f'dbo.{table}') or notebookutils.fs.exists(table_path(TEST_LAKEHOUSE_ID, table)):
        raise RuntimeError(f'{table} already exists; no overwrite is allowed')

frames, table_report, expected_counts = {}, {}, {}
try:
    for entity_name, (table, key, item_id) in EXPECTED_ENTITIES.items():
        source_path = table_path(item_id, table)
        detail = spark.sql(f'DESCRIBE DETAIL delta.`{source_path}`').first().asDict()
        frame = spark.read.format('delta').option('versionAsOf', spark.sql(f'DESCRIBE HISTORY delta.`{source_path}` LIMIT 1').first()['version']).load(source_path).cache()
        frames[table] = frame
        row_count = frame.count()
        if not 0 < row_count <= 50000:
            raise ValueError(f'{table}: expected 1..50000 workshop rows')
        if frame.filter(F.col(key).isNull()).limit(1).count() or frame.select(key).distinct().count() != row_count:
            raise ValueError(f'{table}: null or duplicate primary key')
        table_report[table] = {'key': key, 'source_item_id': item_id, 'rows': row_count,
                               'schema': frame.schema.jsonValue(), 'source_properties': detail['properties']}
        print(table, row_count, frame.dtypes, detail['properties'])
    for name, (origin, target, table, origin_column, target_column) in EXPECTED_RELATIONS.items():
        mapping = frames[table]
        valid = mapping.filter(F.col(origin_column).isNotNull() & F.col(target_column).isNotNull())
        for entity_name, column in ((origin, origin_column), (target, target_column)):
            node_table, node_key, _ = EXPECTED_ENTITIES[entity_name]
            node = frames[node_table]
            if mapping.schema[column].dataType != node.schema[node_key].dataType:
                raise ValueError(f'{name}: physical key types differ')
            missing = mapping.filter(F.col(column).isNotNull()).join(node, mapping[column] == node[node_key], 'left_anti')
            if missing.limit(1).count():
                raise ValueError(f'{name}: non-null reference has no matching node')
        expected_counts[name] = valid.count()
    for table, frame in frames.items():
        frame.write.format('delta').mode('errorifexists').saveAsTable(f'dbo.{table}')
        copied = spark.read.format('delta').load(table_path(TEST_LAKEHOUSE_ID, table))
        if copied.schema != frame.schema or copied.exceptAll(frame).limit(1).count() or frame.exceptAll(copied).limit(1).count():
            raise RuntimeError(f'{table}: copied values/schema differ')
        target_detail = spark.sql(f'DESCRIBE DETAIL delta.`{table_path(TEST_LAKEHOUSE_ID, table)}`').first().asDict()
        if target_detail['properties'].get('delta.columnMapping.mode', 'none') != 'none':
            raise RuntimeError(f'{table}: target has column mapping enabled')
        table_report[table]['target_properties'] = target_detail['properties']
    manifest = dict(recipe, format_version=1, tables=table_report, expected_counts=expected_counts)
    notebookutils.fs.put(SETUP_ROOT + '/manifest.json', json.dumps(manifest, ensure_ascii=False), False)
    print('COPY VERIFIED. TEST_LAKEHOUSE_ID =', TEST_LAKEHOUSE_ID)
    print('Expected relationship counts:', expected_counts)
    print('Manifest:', SETUP_ROOT + '/manifest.json')
finally:
    for frame in frames.values():
        frame.unpersist()
